# Phase 6: Prophet vs XGBoost Demand Forecasting

```
PHASE 6: Prophet vs XGBoost Demand Forecasting

Task: Predict category Google Trends interest_score
      4 weeks ahead
Data: 52 weeks x 10 categories
Split: Train weeks 1-40, Test weeks 41-52 (temporal)
Evaluation: MAE and RMSE per category, then averaged
Leakage guard: features use only past weeks, never current
```


## Section 0 — Setup

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'prophet'], check=False)

import os, pathlib
os.chdir(r'C:\Users\Hp\Desktop\trendshelf')
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = str(pathlib.Path('credentials.json').resolve())

from google.cloud import bigquery
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from prophet import Prophet
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import warnings
warnings.filterwarnings('ignore')

client = bigquery.Client(project='windy-container-451804-n4')
os.makedirs('docs/screenshots', exist_ok=True)

RANDOM_STATE      = 42
FORECAST_HORIZON  = 4
TRAIN_CUTOFF_WEEK = 40
print('Setup complete.')


## Section 1 — Load Data

In [ ]:
query = '''
SELECT
  trend_date,
  category,
  interest_score
FROM `windy-container-451804-n4.bronze.google_trends_raw`
ORDER BY category, trend_date
'''

raw = client.query(query).to_dataframe()
raw['trend_date'] = pd.to_datetime(raw['trend_date'])

print(f'Shape: {raw.shape}  (expected 520)')
print(f'Date range: {raw["trend_date"].min().date()} to {raw["trend_date"].max().date()}')
print()
wk = raw.groupby('category')['trend_date'].count()
print('Weeks per category:')
print(wk.to_string())


## Section 2 — Feature Engineering (XGBoost, no lookahead)

In [ ]:
frames = []

for cat, grp in raw.groupby('category'):
    g = grp.sort_values('trend_date').copy().reset_index(drop=True)

    g['lag_1']          = g['interest_score'].shift(1)
    g['lag_2']          = g['interest_score'].shift(2)
    g['lag_4']          = g['interest_score'].shift(4)
    g['lag_8']          = g['interest_score'].shift(8)
    g['rolling_mean_4'] = g['interest_score'].rolling(4, min_periods=2).mean()
    g['rolling_std_4']  = g['interest_score'].rolling(4, min_periods=2).std()
    g['rolling_mean_8'] = g['interest_score'].rolling(8, min_periods=4).mean()
    g['momentum']       = g['lag_1'] - g['lag_4']
    g['velocity']       = g['lag_1'] - g['lag_2']
    g['week_of_year']   = g['trend_date'].dt.isocalendar().week.astype(int)
    g['month']          = g['trend_date'].dt.month

    # Target: actual interest_score 4 weeks in the future (no lookahead in features)
    g['target_4wk'] = g['interest_score'].shift(-4)

    frames.append(g)

df = pd.concat(frames, ignore_index=True)

# Label encode category
cats = sorted(df['category'].unique())
cat_map = {c: i for i, c in enumerate(cats)}
df['category_encoded'] = df['category'].map(cat_map)

feature_cols = [
    'lag_1', 'lag_2', 'lag_4', 'lag_8',
    'rolling_mean_4', 'rolling_std_4', 'rolling_mean_8',
    'momentum', 'velocity',
    'week_of_year', 'month', 'category_encoded'
]

# Drop rows with NaN in target (last 4 per category) or lag features (first 8)
df = df.dropna(subset=['target_4wk'] + feature_cols).reset_index(drop=True)
df = df.sort_values('trend_date').reset_index(drop=True)

print(f'Final shape after feature engineering: {df.shape}')
print(f'Date range: {df["trend_date"].min().date()} to {df["trend_date"].max().date()}')
print(f'Rows per category: {df.groupby("category").size().min()} min, '
      f'{df.groupby("category").size().max()} max')


## Section 3 — Temporal Train/Test Split

In [ ]:
unique_dates = sorted(raw['trend_date'].unique())  # use raw 52-week range so cutoff lands mid-series
print(f'Unique weeks available after feature engineering: {len(unique_dates)}')

cutoff_date = unique_dates[TRAIN_CUTOFF_WEEK - 1]
print(f'Cutoff date (end of train): {pd.Timestamp(cutoff_date).date()}')

train = df[df['trend_date'] <= cutoff_date].copy()
test  = df[df['trend_date'] >  cutoff_date].copy()

train_med = train[feature_cols].median()
X_train = train[feature_cols].fillna(train_med)
y_train = train['target_4wk']
X_test  = test[feature_cols].fillna(train_med)
y_test  = test['target_4wk']

print(f'Train: {len(train)} rows | {train["trend_date"].min().date()} to {train["trend_date"].max().date()}')
print(f'Test:  {len(test)} rows  | {test["trend_date"].min().date()} to {test["trend_date"].max().date()}')
print(f'X_train: {X_train.shape}  X_test: {X_test.shape}')


## Section 4 — Naive Baseline (predict last known value)

In [ ]:
naive_pred = test['lag_1'].fillna(train_med['lag_1'])
naive_mae  = mean_absolute_error(y_test, naive_pred)
naive_rmse = np.sqrt(mean_squared_error(y_test, naive_pred))

print(f'Naive baseline MAE:  {naive_mae:.2f}')
print(f'Naive baseline RMSE: {naive_rmse:.2f}')
print('(Predicting last known interest_score as the 4-week-ahead forecast)')


## Section 5 — XGBoost Model

In [ ]:
xgb = XGBRegressor(
    n_estimators=200, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    random_state=RANDOM_STATE, verbosity=0
)
xgb.fit(X_train, y_train)
xgb_pred = xgb.predict(X_test)

xgb_mae  = mean_absolute_error(y_test, xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(y_test, xgb_pred))

# Store predictions alongside metadata for per-category breakdown
test = test.copy()
test['xgb_pred'] = xgb_pred

print(f'XGBoost MAE:  {xgb_mae:.2f}')
print(f'XGBoost RMSE: {xgb_rmse:.2f}')
print(f'Beats naive MAE:  {xgb_mae < naive_mae}')
print(f'Beats naive RMSE: {xgb_rmse < naive_rmse}')
print(f'MAE improvement over naive: {(naive_mae - xgb_mae)/naive_mae*100:.1f}%')


## Section 6 — Prophet Model (per category)

In [ ]:
prophet_rows = []

for cat in sorted(raw['category'].unique()):
    cat_raw = raw[raw['category'] == cat].sort_values('trend_date').copy()

    prop_df = cat_raw[['trend_date', 'interest_score']].rename(
        columns={'trend_date': 'ds', 'interest_score': 'y'})

    train_p = prop_df[prop_df['ds'] <= cutoff_date]
    test_p  = prop_df[prop_df['ds'] >  cutoff_date]

    if len(train_p) < 10:
        print(f'Skipping {cat} — only {len(train_p)} train rows')
        continue

    m = Prophet(
        yearly_seasonality=True,
        weekly_seasonality=False,
        daily_seasonality=False,
        seasonality_mode='multiplicative'
    )
    m.fit(train_p)

    # Forecast far enough to cover all test weeks
    future = m.make_future_dataframe(periods=16, freq='W')
    forecast = m.predict(future)

    # Align to test dates
    fc_test = forecast[['ds', 'yhat']].merge(
        test_p, on='ds', how='inner')

    if len(fc_test) == 0:
        print(f'No matching test dates for {cat} — check date alignment')
        continue

    fc_test['category']    = cat
    fc_test['prophet_pred'] = fc_test['yhat'].clip(lower=0)  # interest can't be negative
    fc_test['actual']       = fc_test['y']
    prophet_rows.append(fc_test[['ds', 'category', 'actual', 'prophet_pred']])
    print(f'{cat:<22} train={len(train_p)} rows  test={len(fc_test)} rows')

prophet_all = pd.concat(prophet_rows, ignore_index=True)

prophet_mae  = mean_absolute_error(prophet_all['actual'], prophet_all['prophet_pred'])
prophet_rmse = np.sqrt(mean_squared_error(prophet_all['actual'], prophet_all['prophet_pred']))

print(f'\nProphet MAE:  {prophet_mae:.2f}')
print(f'Prophet RMSE: {prophet_rmse:.2f}')
print(f'Beats naive MAE:  {prophet_mae < naive_mae}')
print(f'MAE improvement over naive: {(naive_mae - prophet_mae)/naive_mae*100:.1f}%')


## Section 7 — Per-Category Breakdown

In [ ]:
rows = []

for cat in sorted(raw['category'].unique()):
    # XGBoost
    t = test[test['category'] == cat]
    if len(t) == 0:
        continue
    xgb_cat_mae  = mean_absolute_error(t['target_4wk'], t['xgb_pred'])
    xgb_cat_rmse = np.sqrt(mean_squared_error(t['target_4wk'], t['xgb_pred']))
    naive_cat_mae = mean_absolute_error(t['target_4wk'], t['lag_1'].fillna(train_med['lag_1']))

    # Prophet
    p = prophet_all[prophet_all['category'] == cat]
    if len(p) == 0:
        prop_mae = np.nan
    else:
        prop_mae = mean_absolute_error(p['actual'], p['prophet_pred'])

    # Determine winner
    valid = {k: v for k, v in {'XGBoost': xgb_cat_mae, 'Prophet': prop_mae}.items()
             if not np.isnan(v)}
    winner = min(valid, key=valid.get) if valid else 'N/A'

    rows.append({
        'category':   cat,
        'naive_mae':  round(naive_cat_mae, 2),
        'xgb_mae':    round(xgb_cat_mae,   2),
        'xgb_rmse':   round(xgb_cat_rmse,  2),
        'prophet_mae': round(prop_mae, 2) if not np.isnan(prop_mae) else np.nan,
        'winner':     winner
    })

cat_df = pd.DataFrame(rows).sort_values('xgb_mae')

print(f'{"Category":<22} {"Naive MAE":>10} {"XGB MAE":>8} {"XGB RMSE":>9} {"Prophet MAE":>12} {"Winner":>10}')
print('-' * 77)
for _, r in cat_df.iterrows():
    pm = f'{r["prophet_mae"]:>12.2f}' if not (isinstance(r['prophet_mae'], float) and np.isnan(r['prophet_mae'])) else f'{"N/A":>12}'
    print(f'{r["category"]:<22} {r["naive_mae"]:>10.2f} {r["xgb_mae"]:>8.2f} {r["xgb_rmse"]:>9.2f} {pm} {r["winner"]:>10}')

easiest = cat_df.iloc[0]['category']
hardest = cat_df.iloc[-1]['category']
xgb_wins = (cat_df['winner'] == 'XGBoost').sum()
prop_wins = (cat_df['winner'] == 'Prophet').sum()
print(f'\nEasiest to forecast: {easiest}  ({cat_df.iloc[0]["xgb_mae"]:.2f} MAE)')
print(f'Hardest to forecast: {hardest}  ({cat_df.iloc[-1]["xgb_mae"]:.2f} MAE)')
print(f'XGBoost wins: {xgb_wins}/10 categories')
print(f'Prophet wins: {prop_wins}/10 categories')


## Section 8 — XGBoost Feature Importance

In [ ]:
fi = pd.Series(xgb.feature_importances_, index=feature_cols).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(fi.index[::-1], fi.values[::-1], color='#4C72B0')
ax.set_xlabel('Importance (gain)')
ax.set_title('XGBoost: Which signals drive 4-week demand forecast?')
plt.tight_layout()
plt.savefig('docs/screenshots/phase6_xgb_feature_importance.png', dpi=120)
plt.show()
print('Saved: docs/screenshots/phase6_xgb_feature_importance.png')

print('\nTop 5 features:')
for feat, imp in fi.head(5).items():
    print(f'  {feat:<25} {imp:.4f}')

top_feature = fi.index[0]


## Section 9 — Leakage Guard (Shuffled Target)

In [ ]:
y_shuffled = y_train.sample(frac=1, random_state=99).reset_index(drop=True)

xgb_shuf = XGBRegressor(
    n_estimators=200, max_depth=4,
    random_state=RANDOM_STATE, verbosity=0)
xgb_shuf.fit(X_train, y_shuffled)
shuf_pred = xgb_shuf.predict(X_test)
shuf_mae  = mean_absolute_error(y_test, shuf_pred)

print(f'Shuffled-target MAE: {shuf_mae:.2f}')
print(f'Real model MAE:      {xgb_mae:.2f}')
print(f'Naive baseline MAE:  {naive_mae:.2f}')
print()
if shuf_mae > xgb_mae * 1.2:
    print('OK: Real model significantly better than shuffled — no leakage')
    print(f'   Improvement: {(shuf_mae - xgb_mae)/shuf_mae*100:.1f}% better than shuffled baseline')
else:
    print('WARNING: Shuffled MAE close to real — weak signal or possible leakage')


## Section 10 — Honest Conclusion

In [ ]:
# Determine overall winner
overall_winner = 'XGBoost' if xgb_mae <= prophet_mae else 'Prophet'

def fmt(v): return f'{v:.2f}'

lines = [
    '# Phase 6: Prophet vs XGBoost Demand Forecasting', '',
    '## Task',
    'Predict Google Trends interest_score 4 weeks ahead per category.', '',
    '## Why this is valid',
    '- Target is FUTURE interest_score (not TrendShelf score)',
    '- XGBoost features are past lags and rolling stats only',
    '- Temporal train/test split — test weeks 41-52 never seen in training',
    '- Shuffled-target leakage guard confirms data integrity', '',
    '## Results',
    '| Model   | MAE  | RMSE | Beats Naive |',
    '|---------|------|------|-------------|',
    '| Naive   | ' + fmt(naive_mae)   + ' | ' + fmt(naive_rmse)   + ' | baseline |',
    '| XGBoost | ' + fmt(xgb_mae)     + ' | ' + fmt(xgb_rmse)     + ' | ' + str(xgb_mae < naive_mae)     + ' |',
    '| Prophet | ' + fmt(prophet_mae) + ' | ' + fmt(prophet_rmse) + ' | ' + str(prophet_mae < naive_mae) + ' |',
    '',
    '## Winner',
    overall_winner + ' wins overall (lower MAE).',
    'Easiest to forecast: ' + easiest + '.  Hardest: ' + hardest + '.', '',
    '## Key finding',
    'Top XGBoost signal: ' + top_feature,
    'XGBoost wins ' + str(xgb_wins) + '/10 categories, Prophet wins ' + str(prop_wins) + '/10.', '',
    '## Leakage guard',
    'Shuffled-target MAE: ' + fmt(shuf_mae) + '  Real model MAE: ' + fmt(xgb_mae),
    ('No leakage — real model clearly beats shuffled baseline.'
     if shuf_mae > xgb_mae * 1.2
     else 'Signal is weak — interpret results conservatively.'), '',
    '## Limitations',
    '- 52 weeks — Prophet needs 2+ seasonal cycles ideally',
    '- Google Trends normalization: 100 = peak week (relative)',
    '- Values may shift when new data is added (rescaling artifact)',
    '- Improve with 12+ months of collection', '',
    '## Why rule-based scoring over ML forecasting',
    'TrendShelf uses rule-based scoring for interpretability.',
    'ML forecasting here validates that the signals used in',
    'scoring have genuine predictive power for demand.',
]

text = '\n'.join(lines)
print(text)

os.makedirs('docs', exist_ok=True)
with open('docs/phase6_model_comparison.md', 'w', encoding='utf-8') as f:
    f.write(text)
print('\nSaved: docs/phase6_model_comparison.md')
